In [1]:
# Medicare Provider Analytics Using Apache Spark

## Notebook 3 – Data Integration

# Course:CS-675 Big Data Management & Analytics

# Author:Judi-Ann Beckford

### Objective

# Join the Medicare Provider Service dataset with the Medicare Enrollment dataset using the National Provider Identifier (NPI) to create a master analytics dataset.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [3]:
spark = (
    SparkSession.builder
    .appName("Medicare Data Integration")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.default.parallelism", "24")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
26/09/01 19:07:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
Spark master: local[6]
Driver memory: 6g


In [4]:
provider_preprocessed_path = "../data/processed/provider_services_preprocessed"

provider_clean_df = spark.read.parquet(provider_preprocessed_path)

print("Preprocessed provider-service dataset loaded.")
print(f"Rows: {provider_clean_df.count():,}")
print(f"Columns: {len(provider_clean_df.columns)}")

provider_clean_df.printSchema()

Preprocessed provider-service dataset loaded.


[Stage 2:===================================================>     (48 + 5) / 53]

Rows: 9,781,673
Columns: 26
root
 |-- NPI: long (nullable = true)
 |-- PROVIDER_SPECIALTY: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- HCPCS_DESCRIPTION: string (nullable = true)
 |-- PLACE_OF_SERVICE: string (nullable = true)
 |-- TOTAL_BENEFICIARIES: double (nullable = true)
 |-- TOTAL_SERVICES: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT: double (nullable = true)
 |-- TOTAL_BENEFICIARIES_CAPPED: double (nullable = true)
 |-- TOTAL_SERVICES_CAPPED: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE_CAPPED: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED_CAPPED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT_CAPPED: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT_CAPPED: double (nullable = true)
 |-- TOTAL_BENEFICIARIES_CAPPED_NORM: double (nullable = true)
 |-- TOTAL_SERVICES_CA

In [6]:
enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"

enrollment_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Enrollment dataset loaded.")
print(f"Enrollment columns: {len(enrollment_raw_df.columns)}")

[Stage 6:============================>                            (12 + 6) / 24]

Enrollment dataset loaded.
Enrollment columns: 11


In [3]:
provider_file = "../data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv"
enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"
output_path = "../data/processed/master_provider_services"

print("Provider file exists:", os.path.exists(provider_file))
print("Enrollment file exists:", os.path.exists(enrollment_file))
print("Current folder:", os.getcwd())

Provider file exists: True
Enrollment file exists: True
Current folder: /Users/judi-annbeckford/Documents/Medicare-Provider-Analytics-Spark/notebooks


In [4]:
provider_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(provider_file)
)

enrollment_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Provider-service columns:", len(provider_raw_df.columns))
print("Enrollment columns:", len(enrollment_raw_df.columns))

[Stage 3:============================>                           (12 + 10) / 24]

Provider-service columns: 28
Enrollment columns: 11


In [30]:
# Load the preprocessed provider-service dataset created in Notebook 02
provider_preprocessed_path = "../data/processed/provider_services_preprocessed"

provider_clean_df = spark.read.parquet(provider_preprocessed_path)

print("Preprocessed provider-service dataset loaded.")
print(f"Rows: {provider_clean_df.count():,}")
print(f"Columns: {len(provider_clean_df.columns)}")

provider_clean_df.printSchema()

Preprocessed provider-service dataset loaded.
Rows: 9,781,673
Columns: 26
root
 |-- NPI: long (nullable = true)
 |-- PROVIDER_SPECIALTY: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- HCPCS_DESCRIPTION: string (nullable = true)
 |-- PLACE_OF_SERVICE: string (nullable = true)
 |-- TOTAL_BENEFICIARIES: double (nullable = true)
 |-- TOTAL_SERVICES: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT: double (nullable = true)
 |-- TOTAL_BENEFICIARIES_CAPPED: double (nullable = true)
 |-- TOTAL_SERVICES_CAPPED: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE_CAPPED: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED_CAPPED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT_CAPPED: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT_CAPPED: double (nullable = true)
 |-- TOTAL_BENEFICIARIES_CAPPED_NORM: d

In [31]:
provider_clean_df = (
    provider_clean_df
    .withColumn(
        "ESTIMATED_TOTAL_MEDICARE_PAYMENT",
        F.col("TOTAL_SERVICES") * F.col("AVG_MEDICARE_PAYMENT")
    )
    .withColumn(
        "ESTIMATED_TOTAL_SUBMITTED_CHARGE",
        F.col("TOTAL_SERVICES") * F.col("AVG_SUBMITTED_CHARGE")
    )
)

In [32]:
provider_clean_df.show(5, truncate=False)
provider_clean_df.printSchema()

+----------+------------------+----------+-------------------------------------------------------------------------------------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+--------------------------+---------------------+---------------------------+---------------------------+---------------------------+-------------------------------+-------------------------------+--------------------------+--------------------------------+--------------------------------+--------------------------------+------------------------------------+----------------------+----------------+--------------+--------------------------------+--------------------------------+
|NPI       |PROVIDER_SPECIALTY|HCPCS_CODE|HCPCS_DESCRIPTION                                                                          |PLACE_OF_SERVICE|TOTAL_BENEFICIARIES|TOTAL_SERVICES|AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAY

In [8]:
enrollment_selected_df = (
    enrollment_raw_df
    .select(
        F.col("NPI").cast("long").alias("NPI"),
        F.trim(F.col("PROVIDER_TYPE_DESC")).alias("ENROLLMENT_PROVIDER_TYPE"),
        F.upper(F.trim(F.col("STATE_CD"))).alias("ENROLLMENT_STATE"),
        F.trim(F.col("FIRST_NAME")).alias("FIRST_NAME"),
        F.trim(F.col("MDL_NAME")).alias("MIDDLE_NAME"),
        F.trim(F.col("LAST_NAME")).alias("LAST_NAME"),
        F.trim(F.col("ORG_NAME")).alias("ORGANIZATION_NAME")
    )
    .filter(F.col("NPI").isNotNull())
)

In [9]:
enrollment_dimension_df = (
    enrollment_selected_df
    .groupBy("NPI")
    .agg(
        F.first("ENROLLMENT_PROVIDER_TYPE", ignorenulls=True)
            .alias("ENROLLMENT_PROVIDER_TYPE"),
        F.first("ENROLLMENT_STATE", ignorenulls=True)
            .alias("ENROLLMENT_STATE"),
        F.first("FIRST_NAME", ignorenulls=True)
            .alias("FIRST_NAME"),
        F.first("MIDDLE_NAME", ignorenulls=True)
            .alias("MIDDLE_NAME"),
        F.first("LAST_NAME", ignorenulls=True)
            .alias("LAST_NAME"),
        F.first("ORGANIZATION_NAME", ignorenulls=True)
            .alias("ORGANIZATION_NAME"),
        F.count("*").alias("ENROLLMENT_RECORD_COUNT"),
        F.countDistinct("ENROLLMENT_PROVIDER_TYPE")
            .alias("DISTINCT_ENROLLMENT_TYPES"),
        F.countDistinct("ENROLLMENT_STATE")
            .alias("DISTINCT_ENROLLMENT_STATES")
    )
)

In [10]:
enrollment_dimension_rows = enrollment_dimension_df.count()
enrollment_dimension_unique_npis = (
    enrollment_dimension_df.select("NPI").distinct().count()
)

print(f"Enrollment dimension rows: {enrollment_dimension_rows:,}")
print(f"Unique NPIs: {enrollment_dimension_unique_npis:,}")
print(
    "NPI is unique:",
    enrollment_dimension_rows == enrollment_dimension_unique_npis
)

[Stage 13:==========================================>             (18 + 6) / 24]

Enrollment dimension rows: 2,556,656
Unique NPIs: 2,556,656
NPI is unique: True


In [11]:
provider_rows_before_join = provider_clean_df.count()
provider_unique_npis = provider_clean_df.select("NPI").distinct().count()

print(f"Provider-service rows before join: {provider_rows_before_join:,}")
print(f"Provider-service unique NPIs: {provider_unique_npis:,}")

[Stage 22:========================================>               (38 + 6) / 53]

Provider-service rows before join: 9,781,673
Provider-service unique NPIs: 1,207,473


In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [13]:
master_df = (
    provider_clean_df
    .join(
        enrollment_dimension_df,
        on="NPI",
        how="left"
    )
)

In [14]:
master_rows = master_df.count()

print(f"Rows before join: {provider_rows_before_join:,}")
print(f"Rows after join:  {master_rows:,}")
print("Row count preserved:", provider_rows_before_join == master_rows)

Rows before join: 9,781,673
Rows after join:  9,781,673
Row count preserved: True


In [15]:
matched_rows = master_df.filter(
    F.col("ENROLLMENT_PROVIDER_TYPE").isNotNull()
).count()

unmatched_rows = master_rows - matched_rows
match_rate = (matched_rows / master_rows) * 100

print(f"Matched service records:   {matched_rows:,}")
print(f"Unmatched service records: {unmatched_rows:,}")
print(f"Enrollment match rate:     {match_rate:.2f}%")

[Stage 35:============================>                           (12 + 6) / 24]

Matched service records:   9,645,673
Unmatched service records: 136,000
Enrollment match rate:     98.61%


In [16]:
master_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (19)
+- Project (18)
   +- SortMergeJoin LeftOuter (17)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan parquet  (1)
      +- SortAggregate (16)
         +- Sort (15)
            +- Exchange (14)
               +- SortAggregate (13)
                  +- SortAggregate (12)
                     +- Sort (11)
                        +- Exchange (10)
                           +- SortAggregate (9)
                              +- Sort (8)
                                 +- Expand (7)
                                    +- Project (6)
                                       +- Filter (5)
                                          +- Scan csv  (4)


(1) Scan parquet 
Output [26]: [NPI#0L, PROVIDER_SPECIALTY#1, HCPCS_CODE#2, HCPCS_DESCRIPTION#3, PLACE_OF_SERVICE#4, TOTAL_BENEFICIARIES#5, TOTAL_SERVICES#6, AVG_SUBMITTED_CHARGE#7, AVG_MEDICARE_ALLOWED#8, AVG_MEDICARE_PAYMENT#9, AVG_STANDARDIZED_PAYMENT#10, TOTAL_BENEFICIARIES_CAPPED#11, TOTAL_S

In [17]:
## Join Performance Tuning

#The initial integration used a broadcast join for the enrollment dimension. During execution, Spark reported insufficient memory to broadcast the approximately 2.56 million enrollment records. To improve reliability and scalability, automatic broadcasting was disabled and the join was executed using Spark's SortMergeJoin strategy. The revised join preserved all 9,781,673 provider-service records and maintained the same 98.61% enrollment match rate, demonstrating that the performance adjustment did not alter the analytical results.

In [18]:
## Join Design Rationale

#The provider-service dataset contains multiple service records for each NPI, while the enrollment source can also contain multiple records for a single NPI. To prevent a many-to-many join from multiplying provider-service rows, the enrollment data was first transformed into a one-row-per-NPI analytical dimension.

#A left join was then used so that all Medicare provider-service records were retained, including records without a corresponding enrollment match.

#Validation confirmed that the provider-service row count remained unchanged after the join.

In [19]:
matched_rows = master_df.filter(
    F.col("ENROLLMENT_PROVIDER_TYPE").isNotNull()
).count()

unmatched_rows = master_rows - matched_rows
match_rate = (matched_rows / master_rows) * 100

print(f"Matched service records:   {matched_rows:,}")
print(f"Unmatched service records: {unmatched_rows:,}")
print(f"Enrollment match rate:     {match_rate:.2f}%")

[Stage 40:==========================================>             (18 + 6) / 24]

Matched service records:   9,645,673
Unmatched service records: 136,000
Enrollment match rate:     98.61%


In [20]:
state_comparison_df = (
    master_df
    .withColumn(
        "STATE_MATCH",
        F.when(
            F.col("PROVIDER_STATE") == F.col("ENROLLMENT_STATE"),
            "MATCH"
        )
        .when(F.col("ENROLLMENT_STATE").isNull(), "NO_ENROLLMENT_MATCH")
        .otherwise("DIFFERENT")
    )
)

state_comparison_df.groupBy("STATE_MATCH").count().show()

[Stage 53:============================>                           (12 + 6) / 24]

+-------------------+-------+
|        STATE_MATCH|  count|
+-------------------+-------+
|          DIFFERENT|1091586|
|              MATCH|8554087|
|NO_ENROLLMENT_MATCH| 136000|
+-------------------+-------+



In [22]:
output_path = "../data/processed/master_provider_services"

print("Output path:", output_path)

Output path: ../data/processed/master_provider_services


In [23]:
(
    state_comparison_df
    .write
    .mode("overwrite")
    .partitionBy("PROVIDER_STATE")
    .parquet(output_path)
)

print("Master dataset saved to:", output_path)

Master dataset saved to: ../data/processed/master_provider_services


In [24]:
master_parquet_df = spark.read.parquet(output_path)

print(f"Saved Parquet rows: {master_parquet_df.count():,}")
print("Saved Parquet columns:", len(master_parquet_df.columns))

Saved Parquet rows: 9,781,673
Saved Parquet columns: 36


In [25]:
required_columns = [
    "NPI",
    "PROVIDER_SPECIALTY",
    "PROVIDER_STATE",
    "TOTAL_SERVICES",
    "TOTAL_SERVICES_CAPPED",
    "TOTAL_SERVICES_CAPPED_NORM",
    "AVG_MEDICARE_PAYMENT",
    "AVG_MEDICARE_PAYMENT_CAPPED",
    "AVG_MEDICARE_PAYMENT_CAPPED_NORM",
    "PLACE_OF_SERVICE_INDEX",
    "UTILIZATION_BAND",
    "ENROLLMENT_PROVIDER_TYPE",
    "ENROLLMENT_STATE",
    "STATE_MATCH"
]

print("=== FINAL MASTER DATASET VALIDATION ===")
print(f"Rows: {master_parquet_df.count():,}")
print(f"Columns: {len(master_parquet_df.columns)}")
print()

for col_name in required_columns:
    print(
        f"{col_name}:",
        "PRESENT" if col_name in master_parquet_df.columns else "MISSING"
    )

=== FINAL MASTER DATASET VALIDATION ===
Rows: 9,781,673
Columns: 36

NPI: PRESENT
PROVIDER_SPECIALTY: PRESENT
PROVIDER_STATE: PRESENT
TOTAL_SERVICES: PRESENT
TOTAL_SERVICES_CAPPED: PRESENT
TOTAL_SERVICES_CAPPED_NORM: PRESENT
AVG_MEDICARE_PAYMENT: PRESENT
AVG_MEDICARE_PAYMENT_CAPPED: PRESENT
AVG_MEDICARE_PAYMENT_CAPPED_NORM: PRESENT
PLACE_OF_SERVICE_INDEX: PRESENT
UTILIZATION_BAND: PRESENT
ENROLLMENT_PROVIDER_TYPE: PRESENT
ENROLLMENT_STATE: PRESENT
STATE_MATCH: PRESENT


In [24]:
state_comparison_df = (
    master_df
    .withColumn(
        "STATE_MATCH",
        F.when(
            F.col("PROVIDER_STATE") == F.col("ENROLLMENT_STATE"),
            "MATCH"
        )
        .when(
            F.col("ENROLLMENT_STATE").isNull(),
            "NO_ENROLLMENT_MATCH"
        )
        .otherwise("DIFFERENT")
    )
)

state_comparison_df.groupBy("STATE_MATCH").count().show()

[Stage 80:==========================================>             (18 + 6) / 24]

+-------------------+-------+
|        STATE_MATCH|  count|
+-------------------+-------+
|          DIFFERENT|1091586|
|              MATCH|8554087|
|NO_ENROLLMENT_MATCH| 136000|
+-------------------+-------+



In [25]:
output_path = "../data/processed/master_provider_services"

(
    state_comparison_df
    .write
    .mode("overwrite")
    .partitionBy("PROVIDER_STATE")
    .parquet(output_path)
)

print("Master dataset successfully saved!")

Master dataset successfully saved!


In [22]:
## Integration Summary

#- Standardized the NPI field across both datasets.
#- Selected the variables required for provider utilization and payment analysis.
#- Created one enrollment dimension record per NPI to prevent row multiplication.
#- Used a left join to preserve all Medicare provider-service records.
#- Validated the join by comparing row counts before and after integration.
#- Calculated the enrollment matching rate.
#- Compared provider-service and enrollment state values.
#- Saved the integrated dataset in compressed, state-partitioned Parquet format.